In [1]:
!pip install -q google-genai faiss-cpu

import faiss
import numpy as ny

from google import genai
from google.genai import types
from google.colab import userdata

client=genai.Client(api_key=userdata.get('Ragproject'))

EMBEDDING_MODEL='gemini-embedding-001'
EMBEDDING_DIMENSION=768

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.3 MB/s eta 0:00:00


In [8]:
def embed(text):
  response=client.models.embed_content(
      model=EMBEDDING_MODEL,
      contents=text,
      config=types.EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION
      )
  )
  return(response.embeddings[0].values)

In [20]:
faq_questions = [
    'How do I reset my password?',
    'How can I change my registered email?',
    'How do I download my invoice?',
    'Why was my payment declined?',
    'How can I cancel my subscription?',
    'How do I update my phone number?'
]

faq_answers = [
    'You can reset your password from Settings > Security > Reset Password.',
    'Go to Account Settings and update your email address.',
    'Open Billing > Invoices and select the required invoice.',
    'Check your card details, available balance, and contact your bank if needed.',
    'Open Subscription Settings and choose Cancel Subscription.',
    'Go to Profile Settings and edit your registered phone number.'
]

#embedding
FAQ_vector = ny.array([embed(q) for q in faq_questions])
print("FAQ Vector shape: ",FAQ_vector.shape)

#Normalzing
FAQ_vector = FAQ_vector/ny.linalg.norm(FAQ_vector, axis=1, keepdims=True)
print("Normalised FAQ Vector shape: ",FAQ_vector.shape)


#SEARCH
def search(query,k_top=1):

    query_vector=embed(query)
    query_vector=query_vector/ny.linalg.norm(query_vector)

    score = FAQ_vector @ query_vector

    top_reply=ny.argsort(score)[::-1][:k_top]

    return[
        {
            'query': query,
            'reply': faq_answers[i],
            'score': float(score[i])
        }
        for i in top_reply
    ]

#TESTING
question=[
    'How can I change my password?',
    'I was charged but my payment failed',
    'How do I bake a chocolate cake?'
]

for i, query in enumerate(question,1):
    print(f"_________TURN{i}_________\n{query}")

    for q in search(query):
        print(f"{q['score']:.3f} - {q['reply']}")



FAQ Vector shape:  (6, 768)
Normalised FAQ Vector shape:  (6, 768)
_________TURN1_________
How can I change my password?
0.751 - You can reset your password from Settings > Security > Reset Password.
_________TURN2_________
I was charged but my payment failed
0.683 - Check your card details, available balance, and contact your bank if needed.
_________TURN3_________
How do I bake a chocolate cake?
0.522 - You can reset your password from Settings > Security > Reset Password.
